# Ayushman Bharat Healthcare Analytics: Analysis of Hospital Accessibility and Services

Course: Data Analytics using Python (CSL-0312) | ITM University Gwalior

Team: Krishna Sharma, Jacky Rayeen, Tamash Sahu | Mentor: Dr. Sanjay Jain

Place `Data_sets_of_data_analytics.xls` in the same folder as this notebook, then run all cells.

## 1. Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Dataset file (for .xls files install xlrd once: pip install xlrd)
df = pd.read_excel("Data_sets_of_data_analytics.xls")
print(df.shape)
df.info()
print(df.isnull().sum())


## 2. Data cleaning and preprocessing

In [ ]:
# Drop empty columns and columns not needed for the analysis
empty = [c for c in df.columns if df[c].isnull().all()]
df = df.drop(columns=empty)
df = df.drop(columns=["Sno", "Hospital Contact", "Specialities Selected",
                      "Upgraded Specialities", "Submitted Date",
                      "Status Updated Date", "Application Status"])
df = df.rename(columns={"Current Specialities": "Specialities",
                        "OnBoarded For Convergence Scheme": "Convergence Scheme"})

# Standardize text
for col in ["Hospital Name", "District", "Hospital Type", "Empanelment Type"]:
    df[col] = df[col].astype("string").str.strip()
df["Hospital Name"] = df["Hospital Name"].str.title()
df["District"] = df["District"].str.title()

# Duplicates (Hospital Id is the unique key)
print("Duplicate Hospital Ids:", df.duplicated(subset=["Hospital Id"]).sum())
df = df.drop_duplicates(subset=["Hospital Id"])

# Missing values
print("Missing Hospital Type:", df["Hospital Type"].isnull().sum())
df["Hospital Type"] = df["Hospital Type"].fillna("Not Specified")

# Derived attributes
codes = df["Specialities"].astype(str).str.replace(" ", "").str.split(",")
df["Num Specialities"] = codes.apply(lambda l: len({x for x in l if x}))
df["M Codes"] = codes.apply(lambda l: len({x for x in l if x.startswith("M")}))
df["S Codes"] = codes.apply(lambda l: len({x for x in l if x.startswith("S")}))

dist_counts = df["District"].value_counts()
q1, q3 = dist_counts.quantile(0.25), dist_counts.quantile(0.75)
n = df["District"].map(dist_counts)
df["Coverage Level"] = np.select([n >= q3, n <= q1], ["High", "Low"], default="Medium")

df.to_csv("ayushman_mp_hospitals_clean.csv", index=False)
print(df.shape)


## 3. Exploratory data analysis

In [ ]:
district_counts = df["District"].value_counts()
print(district_counts.head(10))
print("Districts covered:", df["District"].nunique())

type_counts = df["Hospital Type"].value_counts()
print(type_counts)
print((type_counts / len(df) * 100).round(1))

print(df["Empanelment Type"].value_counts())

print(df["Num Specialities"].describe())
print(df.groupby("Hospital Type")["Num Specialities"].agg(["count", "mean", "median"]).round(2))

# District vs hospital type (group-wise comparison)
district_type = pd.crosstab(df["District"], df["Hospital Type"])
district_type["Total"] = district_type.sum(axis=1)
top8 = district_type.sort_values("Total", ascending=False).head(8)
print(top8)

# Low-coverage districts
print(district_counts[district_counts <= q1])


## 4. Data visualization

In [ ]:
# Figure 1: Top 10 districts
ax = district_counts.head(10).plot(kind="bar", figsize=(9, 5), color="#2F5597")
ax.set_title("Top 10 Districts by Number of Empanelled Hospitals (Madhya Pradesh)")
ax.set_xlabel("District"); ax.set_ylabel("Number of Hospitals")
plt.xticks(rotation=40, ha="right"); plt.tight_layout()
plt.savefig("fig1_top_districts.png", dpi=150); plt.show()

# Figure 2: Hospital type share
type_counts.plot(kind="pie", labels=None, autopct=lambda p: f"{p:.1f}%" if p >= 3 else "",
                   figsize=(6.5, 6), startangle=90, ylabel="")
plt.legend(type_counts.index, loc="lower left", fontsize=8)
plt.title("Hospital Type Distribution"); plt.tight_layout()
plt.savefig("fig2_hospital_type.png", dpi=150); plt.show()

# Figure 3: Distribution of hospitals per district
plt.figure(figsize=(8, 5))
plt.hist(district_counts.values, bins=15, color="#2F5597", edgecolor="white")
plt.title("Distribution of Hospitals per District")
plt.xlabel("Hospitals in a district"); plt.ylabel("Number of districts")
plt.tight_layout(); plt.savefig("fig3_district_hist.png", dpi=150); plt.show()

# Figure 4: Hospital types in the top 8 districts
top8.drop(columns="Total").plot(kind="bar", stacked=True, figsize=(9, 5.5))
plt.title("Hospital Types in the Top 8 Districts")
plt.xlabel("District"); plt.ylabel("Number of Hospitals")
plt.xticks(rotation=30, ha="right"); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig("fig4_district_type.png", dpi=150); plt.show()


## 5. Statistical analysis (basic)

In [ ]:
counts = district_counts.values.astype(float)

print("Mean   :", round(counts.mean(), 2))
print("Median :", np.median(counts))
print("Mode   :", district_counts.mode().iloc[0])
print("Std Dev:", round(counts.std(ddof=1), 2))
print("CV (%) :", round(counts.std(ddof=1) / counts.mean() * 100, 1))
print("Top 3 share (%):", round(counts[:3].sum() / counts.sum() * 100, 1))

# Gini coefficient (0 = equal, 1 = very unequal)
x = np.sort(counts)
k = len(x)
gini = (2 * np.sum(np.arange(1, k + 1) * x) / (k * x.sum())) - (k + 1) / k
print("Gini   :", round(gini, 3))
